## Actividad 1 instalación de biblioteca datasets

esta biblioteca permite cargar conjuntos de datos Hugging Face de manera sencilla

In [ ]:
# importar la función load_dataset de la biblioteca datasets

from datasets import load_dataset

# cargar el conjunto de datos IMDb Movie reviews
dataset = load_dataset("imdb")

### Convertir el conjunto de datos en un DataFrame de Pandas

Convertimos el conjunto de datos en un DataFrame para facilitar su manipulación y
seleccionamos una muestra del 20% de los datos para agilizar el procesamiento en el
laboratorio.

In [ ]:
import pandas as pd

data = pd.DataFrame(dataset['train'])

# tomar una muestra del 20% del conjunto de datos para reducir el tiempo de procesamiento

data = data.sample(frac=0.2, random_state=42)

### preprocesar el texto y las etiquetas
Se define el texto de las reseñas como las características (X) y las etiquetas de sentimiento (1
= positivo, 0 = negativo)


In [ ]:
# definir las carateristicas (x) y etiquetas (y) para el analisis de sentimiento.
x = data['text'].fillna('') # asegurar que no haya NaN en el texto

y = data['label'] #etiqueta de sentimiento: 1 = positivo, 0 = negativo

### Dividir el conjunto de datos en entrenamiento y prueba
Dividimos el conjunto de datos en entrenamiento y prueba utilizando una proporción de 80%
para entrenamiento y 20% para prueba.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

### Vectorizar el texto utilizamdo TF-IDF

Se convierte el texto en una representación numérica usando TF-IDF, limitando el vocabulario
a las 3000 palabras más frecuentes para reducir la dimensionalidad.

In [ ]:
# Vectorización del texto con TF-IDF para convertirlo en una representación numérica
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=3000) # Limitar el vocabulario a las 3000 palabras más frecuentes

X_train_tfidf = vectorizer.fit_transform(X_train) # Ajustar y transformar el conjunto de entrenamiento

X_test_tfidf = vectorizer.transform(X_test) # Transformar el conjunto de prueba con el mismo vectorizador



### Entrenar el modelo de Regresión logística

Se entrena un modelo de Regresión logística para clasificar las reseñas en positivas o negativas

In [ ]:
# Entrenar un modelo de Regresión Logística en el conjunto de entrenamiento
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000) # Configurar el modelo con un máximo de 1000 iteraciones

model.fit(X_train_tfidf, y_train) # Entrenar el modelo con los datos de entrenamiento vectorizados

### Evaluar el modelo en el conjunto de prueba
Para verificar el rendimiento del modelo, calculamos la precisión en el conjunto de prueba

In [ ]:
# evaluar el modelo calculando la precisión en el conjunto de prueba

from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_tfidf) # Realizar predicciones en el conjunto de pruebas

accuracy = accuracy_score(y_test, y_pred)  # calcular la precisión 

print(f"Accuracy del modelo: {accuracy * 100:.28}%") #Imprimir la precisión en porcentaje

### Conclusión sobre la Evaluación del Modelo
El resultado de precisión del modelo, con un 85.50% en el conjunto de prueba de IMDb
Movie Reviews, indica un rendimiento robusto en la clasificación de reseñas de películas
como positivas o negativas. Este porcentaje sugiere que el modelo es capaz de identificar
correctamente el sentimiento en la mayoría de los casos, lo que demuestra que ha aprendido
a reconocer patrones relevantes en el texto, como términos o expresiones típicas de opiniones
favorables o desfavorables. Una precisión del 85.50% es un buen indicador en el contexto del
análisis de sentimientos, especialmente considerando la diversidad de estilos y longitudes
presentes en las reseñas del conjunto de datos. Este nivel de precisión implica que el modelo
es aplicable para tareas prácticas de análisis de opiniones, aunque aún puede presentar
errores en casos ambiguos o de sentimientos mixtos, lo cual podría ser optimizado con
técnicas adicionales de afinamiento o interpretabilidad.


## Actividad 2: Interpretabilidad de resultados co LME

Aplicar la técnica de LIME (Local Interpretable Model-agnostic Explanations) para
interpretar los resultados de un modelo de análisis de sentimientos en un conjunto de datos de
reseñas de películas. LIME ayuda a identificar las palabras clave que más influyen en la
clasificación del modelo, lo cual permite entender cómo y por qué el modelo toma decisiones
específicas.



In [ ]:
3 # Importación de bibliotecas 

import pandas
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from datasets import load_dataset #Importar Hugging Face Datasets
from lime.lime_text import LimeTextExplainer # Importar LimeTextExplainer para interpretabilidad

### Cargar el conjunto de datos IMDb Movie Reviews desde Hugging Face

Utilizamos el conjunto de datos de IMDb Movie Reviews, que contiene reseñas de películas
etiquetadas como positivas o negativas, y lo convertimos en un DataFrame de pandas para
facilitar su manipulación


In [ ]:
dataset = load_dataset("imdb")

#convertir el conjunto de datos de IMDb en un DataFrame de pandas

data= pd.DataFrame(dataset['train'])


### Tomar una Muestra del Conjunto de Datos
Tomamos una muestra del 20% del conjunto de datos original para reducir el
tiempo de procesamiento.

In [ ]:
data = data.sample(frac=0.2, random_state=42)

### Preprocesamiento de los Datos
Definimos las características (X) y etiquetas (y) a partir de las columnas text y
label del DataFrame. Luego, dividimos los datos en entrenamiento y prueba.

In [ ]:
# definir las características y etiquetas para el análisis de sentimientos

X = data['text'].fillna('') # Reemplazar valores NaN en el texto por cadenas de valores.

y = data['label'] # Etiqueta de sentimiento

# Dividir el conjunto de datos en conjuntos de entrenamiento y prueba

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Vectorización del Texto con TF-IDF
Convertimos el texto en una representación numérica utilizando TfidfVectorizer.
Limitamos el vocabulario a las 3000 palabras más frecuentes para reducir la dimensionalidad.

In [ ]:
vectorizer = TfidfVectorizer(max_features=3000) # Limitar el vocabulario a 3000 palabras 

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### Entrenar el Modelo de Regresión Logística
Entrenamos un modelo de Regresión Logística utilizando los datos de
entrenamiento vectorizados. Este modelo se usará para clasificar el sentimiento en las
reseñas

In [ ]:
# Entrenamiento del modelo de Regresión Logística

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

### Configuración del Explicador de LIME
Creamos un explicador de LIME para texto (LimeTextExplainer) y definimos las
clases como Negative y Positive para facilitar la interpretación de los resultados.

In [ ]:
# configuración del explicador de LIME para texto

explainer = LimeTextExplainer(class_names=["Negative", "Positive"])

### Seleccionar una Reseña de Prueba para Interpretación
Seleccionamos una reseña del conjunto de prueba para interpretar su
predicción. En este caso, seleccionamos la reseña en el índice 15.

In [ ]:
# Seleccionar una reseña del conjunto de prueba para la interpretación 

i = 15 # Índice de la reseña a interpretar

texto_prueba = X_test.iloc[i]

### Crear una Función para Predicción de Probabilidad
Definimos una función envolvente que toma un texto, lo convierte en su
representación TF-IDF y luego utiliza el modelo para hacer predicciones de probabilidad.

In [ ]:
#función para realizar predicciones de probabilidad

def predict_proba_text(texts):
    texts_tfidf = vectorizer.transform(texts) # convertir el texto a TF-IDF
    return model.predict_proba(texts_tfidf) #realizar la predicción de probabilidad


### Generar la Explicación de LIME para la Reseña Seleccionada
Usamos LIME para generar una explicación de la predicción de sentimiento
para la reseña seleccionada. LIME muestra las palabras más influyentes en la decisión del
modelo.

In [ ]:
# Generar la explicación de LIME para la reseña seleccionada
exp = explainer.explain_instance(texto_prueba, predict_proba_text, num_features=10)

### Visualizar los Resultados de LIME
Descripción: Visualizamos los resultados de LIME en el notebook. Esta visualización destaca
las palabras clave y muestra cómo influyen en la predicción de sentimiento.

In [ ]:
# Visualización de los resultados LIME
exp.show_in_notebook(text=texto_prueba)

La visualización de los resultados generada por LIME muestra una interpretación de la
predicción realizada por el modelo de análisis de sentimientos para la reseña seleccionada.

En esta visualización, cada palabra de la reseña tiene un color que indica su contribución al
sentimiento previsto: las palabras en azul contribuyen a una predicción negativa (Negative),
mientras que las palabras en naranja contribuyen a una predicción positiva (Positive).
En este caso, la probabilidad de que la reseña sea positiva es de 0.77 (77%), mostrada en
naranja, y la probabilidad de que sea negativa es de 0.23 (23%), mostrada en azul. Esto
significa que el modelo ha clasificado esta reseña como positiva, considerando que el 77% de
la predicción se debe a palabras o frases que el modelo asocia con sentimientos favorables.
LIME identifica y resalta las palabras más influyentes en cada dirección, proporcionando una
explicación visual clara de cómo cada término influye en el resultado final. Esto facilita
entender la lógica detrás de la predicción del modelo, identificando los términos clave que
guían la clasificación.

### Actividad 3: Interpretabilidad de Resultados con SHAP

Aplicar SHAP para interpretar los resultados de un modelo de análisis de sentimientos. SHAP,
basado en teoría de juegos, permite entender el impacto de cada palabra en la predicción de
una reseña, proporcionando una visión clara y detallada de cómo el modelo pondera los
términos en sus decisiones.

In [ ]:
# Importar librerias

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from datasets import load_dataset
import shap

### Cargar y Preparar el Conjunto de Datos IMDb

Cargamos el conjunto de datos IMDb desde Hugging Face, tomamos una muestra del 20%,
eliminamos valores nulos y dividimos los datos en conjuntos de entrenamiento y prueba.
Luego, vectorizamos el texto utilizando TF-IDF para convertir el texto en representaciones
numéricas

In [ ]:
# Cargar el conjunto de datos IMDb desde Hugging Face
dataset = load_dataset("imdb")
data = pd.DataFrame(dataset['train'])
data = data.sample(frac=0.2, random_state=42) # Tomar una muestra del 20%

# Definir carateristísticas y etiquetas
X = data['text'].fillna('') # asegurarse de que no haya valores NaN
y = data['label'] #etiqueta de sentimiento

# Dividir el conjunto de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorización del texto con TF-IDF
vectorizer = TfidfVectorizer(max_features=3000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### Entrenar el Modelo de Regresión Logística
Entrenamos un modelo de regresión logística con los datos de entrenamiento vectorizados.
Este modelo se usará para predecir el sentimiento en las reseñas de prueba.

In [ ]:
# entrenar el modelo de Regresió Logística
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

### Configuración del Explicador de SHAP con KernelExplainer
Creamos un explicador SHAP utilizando KernelExplainer, que es adecuado para modelos que
no son completamente interpretables. Utilizamos una muestra de los datos de entrenamiento
para configurar el explicador.

In [ ]:
# Configuración del explicador de SHAP usando el KernelExplainer

background = X_train_tfidf[:100].toarray() 

explainer = shap.KernelExplainer(model.predict_proba, background)

In [ ]:
import shap

# Explicador para modelo lineal (LogisticRegression) con TF-IDF sparse
explainer = shap.LinearExplainer(model, X_train_tfidf, feature_perturbation="interventional")

# Selección de la reseña a interpretar (sparse)
i = 15
X_test_sample = X_test_tfidf[i:i+1]  # sparse, forma (1, n_features)

# Calcular valores SHAP
shap_values = explainer.shap_values(X_test_sample)

# Inicializar JS para visualización
shap.initjs()

# Vocabulario invertido para mostrar palabras
vector_palabras = [word for word, index in sorted(vectorizer.vocabulary_.items(), key=lambda x: x[1])]

# Visualización interactiva
shap.force_plot(
    explainer.expected_value,   # <-- ya es escalar o 1D
    shap_values,                # <-- ya es 1D para la muestra
    X_test_sample.toarray()[0], # vector de la reseña
    feature_names=vector_palabras
)

La visualización de SHAP en este paso muestra cómo cada palabra en la reseña afecta la
predicción de sentimiento del modelo. Las palabras se muestran en lugar de los índices,
permitiendo una interpretación clara de qué términos específicos están influyendo en la
predicción del modelo y si están empujando la predicción hacia un sentimiento positivo o
negativo. Las palabras en rojo contribuyen positivamente al sentimiento predicho, mientras
que las palabras en azul tienen un impacto negativo.